# Tutorial 4: DOE Planning (`grid` vs `sobol`)

Estimated time: 25-40 minutes

## Prerequisites
- The `tutorials` extra, for matplotlib and numpy: `pip install -e '.[tutorials]'`.
  (A core install carries only pydantic, requests and scipy.)
- **No optional backend** (`pymc`, `sbi`) is needed for this notebook. If you are working
  through the whole series, the `py314_bayesmm` environment (`environment.yml`)
  has every backend in one place.
- **Nothing from an earlier tutorial has to be on disk.** Step 2 runs both sweeps itself,
  so T4 stands alone even if you skipped T1.

## Learning aims
- Primary package aim: compare DOE strategies and understand what `plan` gives you
- Secondary scientific aim: *define* coverage, *measure* it, and find the point where a
  measure of coverage stops being a measure of information

## Success criteria
- you can justify a DOE strategy for a given model dimensionality and run budget
- you can say what `scramble` does, and name the case where changing `seed` does nothing

## Why this tutorial matters

DOE strategy is a **budget question**: given N runs you can afford to execute, where do you put them in input space? Two strategies in this curriculum:

- **`grid`** — Cartesian product of per-input value lists. Predictable, easy to interpret, and every coordinate is chosen by you. Great when you want a clean response surface in 1-3 dimensions.
- **`sobol`** — a *low-discrepancy*, space-filling sequence. The same N points spread more evenly over the input box than a random sample, and far more evenly than a regular grid once you are past a few dimensions.

"Spread more evenly" is the kind of claim tutorials normally ask you to take on faith. We won't. **Discrepancy** is a number you can compute for any finite design — it measures how far the design's empirical distribution sits from a perfectly uniform one — and this notebook computes it for the two designs you are about to plan, at 2, 3, 4, 5 and 6 dimensions. You will also meet the point where that number quietly stops answering the question you care about.

## Step 1: Plan both designs — nothing is executed

Both specs place **9 points** on the same box, `a, b ∈ [0, 2]`: a 3×3 grid, and 9 Sobol points. Nine matches the grid point-for-point so the comparison happens at *equal budget*, and it is deliberately the wrong number for Sobol — scipy will say so out loud in the output below, and that warning is one of this tutorial's lessons rather than noise to scroll past.

In [ ]:
# Cross-platform setup (Windows / macOS / Linux) — no shell, no PYTHONPATH prefix.
# Find the repo root so `src/` is importable, then load the shared tutorial helpers.
import sys
from pathlib import Path

_root = Path.cwd().resolve()
while not (_root / "src" / "bayesian_metamodeling").is_dir() and _root != _root.parent:
    _root = _root.parent
if str(_root / "src") not in sys.path:
    sys.path.insert(0, str(_root / "src"))

from bayesian_metamodeling.tutorial import bootstrap, run_mm_cli, run_tool

root = bootstrap()  # chdir to repo root + ensure src/ on sys.path (idempotent)
ROOT = root
print("Repo root:", root)

In [ ]:
run_mm_cli('plan', 'tutorials/specs/model.toy.grid.json')
run_mm_cli('plan', 'tutorials/specs/model.toy.sobol.json')

### Read that output before moving on

Three things are on screen that are routinely skipped.

1. **`Preview:` shows 5 points; the header shows the design size.** `render_plan_preview`
   truncates to the first 5 (`designs/planner.py`), so `Plan points: 9` is the real number.
   Never budget from the preview.
2. **Grid order is alphabetical by variable, outer loop first.** `plan_grid_points` does
   `sorted(design.grid)` then `itertools.product`, so `a` is the outer loop and `b` cycles
   fastest: `(0,0), (0,1), (0,2), (1,0), …`. That is the order the runner executes in — so a
   grid sweep you kill a third of the way through has covered every `b` at `a = 0` and
   nothing anywhere else. **A partial grid sweep is a slice, not a sample.** A partial Sobol
   sweep, by contrast, is a coarser but still space-filling design — a real operational
   advantage on a queue that can pre-empt you.
3. **scipy warned about the Sobol design.** `UserWarning: The balance properties of Sobol'
   points require n to be a power of 2` — because we asked for 9. You still get exactly 9
   points; Step 3 says what is actually lost.

**`plan` is read-only.** It enumerates the DOE points a spec will produce without executing anything — no subprocess, no model, no storage write. `plan_points(spec)` is a pure function of the spec, which is why planning a 4096-run campaign costs milliseconds. Use it for budgeting *before* you spend compute: a 6-input grid at 4 levels each is `4⁶ = 4096` runs, and you learn that instantly instead of three days in.

Whether a Sobol design at some smaller N would serve you as well is a question with an actual answer, not a rule of thumb. Step 4 measures it.

### The two strategies read different parts of the spec

This is the single most useful framework fact in this tutorial, and it is invisible unless someone points at it:

- `plan_sobol_points` derives its box from `io_schema.inputs[].support` and **raises** if any
  input is missing it. For Sobol, the declared domain *is* the design space.
- `plan_grid_points` reads only `design.grid`. It **never consults `support`**.

So a grid design can sample outside the domain the spec itself declares, and nothing complains. Run the cell and watch it happen.

In [ ]:
import copy
import json

from bayesian_metamodeling.designs.planner import DOEPlanError, plan_points
from bayesian_metamodeling.spec import ModelSpec, load_and_validate_modelspec

grid_spec = json.loads((ROOT / "tutorials/specs/model.toy.grid.json").read_text())
sobol_spec = json.loads((ROOT / "tutorials/specs/model.toy.sobol.json").read_text())
print("declared support for a:", grid_spec["io_schema"]["inputs"][0]["support"])

# A grid whose levels sit far outside the declared support.
rogue = copy.deepcopy(grid_spec)
rogue["design"]["grid"]["a"] = [-50.0, 0.0, 99.0]
load_and_validate_modelspec(rogue)  # the validator is happy
rogue_points = plan_points(ModelSpec.model_validate(rogue))
print(f"rogue grid validates AND plans: {len(rogue_points)} points")
print("  a levels planned:", sorted({p["a"] for p in rogue_points}), "<- outside [0.0, 2.0]")

# The same mistake is structurally impossible for sobol: drop support and it refuses.
no_support = copy.deepcopy(sobol_spec)
no_support["io_schema"]["inputs"][0].pop("support")
try:
    plan_points(ModelSpec.model_validate(no_support))
except DOEPlanError as exc:
    print("sobol without support ->", f"{type(exc).__name__}: {exc}")

**The rule:** with `sobol`, the box is the support and the framework enforces it. With `grid`, *you* are responsible for keeping your levels inside the domain you declared — nothing checks it, at plan time or at run time. The failure is silent and it propagates: those out-of-domain rows land in `sweep_rows.csv`, a surrogate gets fitted on them in T5, and a metamodel integrates over them in module 7, all without a single warning. Consider it your job on every grid spec you write.

## Step 2: Execute both designs — the plan is a contract

Note what we are *not* doing: the figure in Step 3 does not need these runs. Design points exist the moment you have a spec — `plan_points` is pure — so the whole comparison below could be drawn without launching a single subprocess. That separation is why `plan` and `run` are different commands.

We run both sweeps anyway, for a reason worth stating: **the plan is a contract, and the self-check at the bottom of this notebook verifies the runner honoured it** — that the `a`/`b` columns landed in `sweep_rows.csv` are exactly the points `plan_points` promised, to the last bit. Eighteen sub-second toy runs; the cost is negligible and it makes T4 self-sufficient whether or not you did T1.

In [ ]:
run_mm_cli("run", "tutorials/specs/model.toy.grid.json")
run_mm_cli("run", "tutorials/specs/model.toy.sobol.json")

## Step 3: See the two designs

**Predict before you run.** Both DOEs place 9 points on the box `a, b ∈ [0, 2]`.

- The grid is a tidy 3×3 lattice on `{0, 1, 2}`. You know exactly where every point falls — including how many sit on the *edge* of the box.
- Sobol will not be a lattice. Its guarantee is dyadic: each successive block of `2^m` points puts exactly one point in every dyadic sub-box at that resolution. With 9 points — one past `2³` — that guarantee is already broken, which is precisely what scipy warned about. So look for near-uniform spread, **not** a Latin-square pattern. Two Sobol points may share a row; some strip of the box may hold none — the next cell bins `a` into 9 equal strips and prints the counts, so you can see for yourself.

Commit to two numbers before running the next cell: of the 9 points, how many touch the boundary of the box — for the **grid**, and for **Sobol**?

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np

LO, HI = 0.0, 2.0  # io_schema.inputs[].support for both a and b

with warnings.catch_warnings():  # the power-of-2 warning; we met it in Step 1
    warnings.simplefilter("ignore")
    sobol_points = plan_points(ModelSpec.model_validate(sobol_spec))
grid_points = plan_points(ModelSpec.model_validate(grid_spec))

G = np.array([[p["a"], p["b"]] for p in grid_points])
S = np.array([[p["a"], p["b"]] for p in sobol_points])


def on_boundary(P):
    return ((P <= LO) | (P >= HI)).any(axis=1)


print(f"grid : {len(G)} points, {int(on_boundary(G).sum())} of them on the boundary of the box")
print(f"sobol: {len(S)} points, {int(on_boundary(S).sum())} of them on the boundary of the box")

# Sobol's 9 points binned into 9 equal strips of `a` — the stratification it does NOT promise.
strips = np.bincount(np.digitize(S[:, 0], np.linspace(LO, HI, 10)[1:-1]), minlength=9)
print("sobol points per equal-width strip of a:", strips.tolist())

fig, axes = plt.subplots(1, 2, figsize=(10, 5), sharex=True, sharey=True)
panels = ((axes[0], G, "grid: 3x3 lattice", "tab:blue"),
          (axes[1], S, "sobol: 9 scrambled points", "tab:orange"))
for ax, P, title, colour in panels:
    ax.add_patch(plt.Rectangle((LO, LO), HI - LO, HI - LO, fill=False, ls="--", ec="0.4"))
    ax.scatter(P[:, 0], P[:, 1], s=90, c=colour, zorder=3)
    ax.set(title=title, xlabel="a", ylabel="b",
           xlim=(LO - 0.15, HI + 0.15), ylim=(LO - 0.15, HI + 0.15))
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

Both panels are drawn on the same axes with the declared support box dashed, so the comparison is honest — autoscaled panels would have silently rescaled the Sobol design to look like it filled the box.

The counts are the thing to take away. **8 of the 9 grid points lie on the boundary; exactly one, `(1, 1)`, is interior.** Not one Sobol point touches the edge. In general an `L`-level grid is interior only where every coordinate avoids the two end levels, so the face fraction is `1 - ((L-2)/L)ᵈ` — here `1 - (1/3)² = 8/9`. Step 4 evaluates it for the 4-level grid, where it becomes `1 - 2⁻ᵈ` and climbs fast with dimension.

The strip counts settle the second half of the prediction. A **Latin hypercube** of 9 points *guarantees* exactly one point in each of the 9 equal-width strips of `a` — and guarantees nothing whatever about the joint spread, so an LHS can legally be a diagonal line. **Sobol makes the opposite trade**: it constrains the joint distribution over dyadic boxes and promises nothing about equal-width marginal strips. With this seed the counts are visibly not all ones. Change the scramble seed and they occasionally do come out all ones — by luck, not by construction, and that distinction between an observed pattern and a guaranteed one is worth more than either design.

## Step 4: Measure coverage instead of asserting it

"Coverage" is used loosely everywhere, including in the first paragraph of this notebook. Two quantities make it precise, and they answer different questions:

- **Discrepancy** — how far the design's empirical distribution is from uniform over the box. `scipy.stats.qmc.discrepancy` returns the centered L2 version; lower is better, and it is the quantity Sobol sequences are *designed* to minimise. It is what you want when the design will be used to average something over the box.
- **Fill distance** — the largest distance from any point of the box to its nearest design point (the covering radius). Lower is better. This is the one a surrogate cares about: it bounds how far the model ever has to interpolate.

Compute both for the two 9-point designs. Predict which way each goes before you look.

In [ ]:
import itertools

from scipy.stats.qmc import Sobol, discrepancy


def to_unit(P):
    """Map the box [0, 2]^2 onto the unit square — `discrepancy` is defined there."""
    return (P - LO) / (HI - LO)


_mesh = np.linspace(0.0, 1.0, 401)
_probe = np.stack(np.meshgrid(_mesh, _mesh, indexing="ij"), axis=-1).reshape(-1, 2)


def fill_distance_2d(P):
    """Worst-case distance from anywhere in the unit square to the nearest design point."""
    d = np.sqrt(((_probe[:, None, :] - to_unit(P)[None, :, :]) ** 2).sum(-1)).min(axis=1)
    return float(d.max())


d_grid, d_sobol = discrepancy(to_unit(G)), discrepancy(to_unit(S))
f_grid, f_sobol = fill_distance_2d(G), fill_distance_2d(S)
print(f"N = 9, d = 2      {'grid':>10} {'sobol':>10}")
print(f"  discrepancy     {d_grid:10.5f} {d_sobol:10.5f}   sobol better by {d_grid / d_sobol:.1f}x")
print(f"  fill distance   {f_grid:10.4f} {f_sobol:10.4f}   sobol better by {f_grid / f_sobol:.2f}x")

Read those two lines together, and note that they disagree about how big the win is. On **discrepancy** Sobol wins by a wide margin (the ratio is printed above). On **fill distance** the two designs are within a couple of percent of each other — a regular lattice is close to optimal for covering radius in low dimension, which is exactly why nobody needs Sobol to study a 2-input model.

The lesson is not that one number is right. It is that **"coverage" is not one quantity**, and the design you prefer depends on which one your downstream use actually consumes. Averaging over the box? Discrepancy. Fitting a surrogate you will query anywhere inside the box? Fill distance.

That is the honest version of "don't be fooled by the 2-D picture". The 2-D picture is not misleading you about Sobol's advantage being small here; the advantage really *is* small here. What it cannot show you is what happens next.

In [ ]:
# Equal budget at every dimension: a 4-level grid (4^d runs) vs the same number of Sobol points.
head = f"{'d':>2} {'grid N = 4^d':>13} {'disc(grid)':>12} {'disc(sobol)':>12} {'ratio':>8} {'grid on a face':>16}"
print(head)
print("-" * len(head))
for d in (2, 3, 4, 5, 6):
    levels = np.linspace(0.0, 1.0, 4)
    Gd = np.array(list(itertools.product(*[levels] * d)))
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        Sd = Sobol(d=d, scramble=True, seed=123).random(len(Gd))
    dg, ds = discrepancy(Gd), discrepancy(Sd)
    face = ((Gd == 0.0) | (Gd == 1.0)).any(axis=1).mean()
    print(f"{d:>2} {len(Gd):>13} {dg:>12.6f} {ds:>12.6f} {dg / ds:>7.0f}x {face:>15.1%}")

print()
print("Smallest power-of-two Sobol N whose discrepancy already beats the 4-level grid:")
for d in (2, 3, 4, 5, 6):
    levels = np.linspace(0.0, 1.0, 4)
    Gd = np.array(list(itertools.product(*[levels] * d)))
    target, n = discrepancy(Gd), 1
    while n <= len(Gd):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            Sd = Sobol(d=d, scramble=True, seed=123).random(n)
        if discrepancy(Sd) <= target:
            break
        n *= 2
    print(f"  d={d}: grid needs {len(Gd):>5} runs; sobol matches its uniformity at {n:>4}")
print("  ...and that second block is a trap. See the text below.")

## The actual lesson: the curse of dimensionality

The table above is the evidence the rest of this notebook was asking you to assume.

- **At equal budget, uniformity diverges.** The grid's discrepancy *rises* with `d` while
  Sobol's *falls*. The ratio in the fifth column is not a rule of thumb — the cell above
  computed it from the designs themselves.
- **A grid spends its budget on the shell.** A 4-level grid point is interior only if *every*
  coordinate is one of the two interior levels, so the interior fraction is `2⁻ᵈ` and the
  face fraction is `1 - 2⁻ᵈ`: 75% at `d = 2`, **98.4% at `d = 6`** — 4032 of your 4096 runs
  spent on the boundary. You already saw the 2-D version in Step 3: 8 of 9 grid points, 0 of
  9 Sobol points.
- **The shell is the worst place to spend it** — but not for the reason usually given. It is
  *not* that "the model behaves predictably at the extremes": in biology the extremes are
  exactly where saturation, switching and cooperativity live. It is that a corner point has
  no neighbours on the far side, so a surrogate fitted there must **extrapolate** and you have
  nothing to check it against, while the interior is where the response surface you will
  later integrate over actually lives. A grid spends most of its budget where the fit is
  least testable.

**Now the trap.** The second block says Sobol needs only 8 points to beat a 4096-point grid on discrepancy at `d = 6`. That is arithmetically true and operationally nonsense. Discrepancy scores how *evenly* a design fills the box; it says nothing about whether the design carries enough information to fit anything — 8 points cannot pin down a 6-input response surface no matter how prettily they are arranged. Uniformity is necessary, not sufficient. **Use discrepancy to decide where points go at a budget you fixed on other grounds; never use it to fix the budget.**

## Step 5: `scramble`, `seed`, and what "deterministic" actually means

The Sobol spec you have been running says `"scramble": true`, and that flag is doing more than it looks like.

- The **base Sobol sequence is deterministic and seed-independent**. It is a fixed
  mathematical object; point 7 of the 2-D sequence is always the same point.
- **`scramble: true` applies Owen scrambling** — a seeded random permutation of the digits
  of each coordinate. It preserves the low-discrepancy property while decorrelating the
  points from the lattice-like artefacts of the raw sequence. It is also what makes error
  estimation possible: with several independent scrambles of the same design you get a
  spread of answers, and therefore an error bar on whatever you integrated.
- **`seed` is read only when `scramble` is true.** `plan_sobol_points` defaults `scramble` to
  `False`, and with an unscrambled engine scipy ignores the seed entirely.

That last point is a trap you can walk into silently: copy this pattern into a spec without `scramble`, change the seed to get a fresh design, and you get the identical design back. Verify it rather than believing it.

In [ ]:
def plan_sobol(n_points, scramble, seed):
    spec = copy.deepcopy(sobol_spec)
    spec["design"]["sobol"] = {"n_points": n_points, "scramble": scramble, "seed": seed}
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        pts = plan_points(ModelSpec.model_validate(spec))
    return [(round(p["a"], 4), round(p["b"], 4)) for p in pts]


for scramble in (False, True):
    for seed in (123, 999):
        print(f"scramble={scramble!s:<5} seed={seed}: {plan_sobol(8, scramble, seed)[:4]}")

print()
print("changing seed changes an UNSCRAMBLED design:",
      plan_sobol(8, False, 123) != plan_sobol(8, False, 999))
print("changing seed changes a SCRAMBLED design:   ",
      plan_sobol(8, True, 123) != plan_sobol(8, True, 999))
print("a scrambled plan is still reproducible:     ",
      plan_sobol(8, True, 123) == plan_sobol(8, True, 123))

So "Sobol is deterministic" is two claims wearing one coat, and conflating them is the standard mistake:

- **Reproducible** — yes, always. Same spec, same points, on any machine, no seed-pinning
  gymnastics. That is what provenance needs and it holds with or without scrambling.
- **Not random** — only when `scramble` is false. A scrambled Sobol design *is* a randomised
  QMC sample; it just happens to be a reproducible one because you fixed the seed.

The spec in this tutorial is scrambled, so "change the `seed` to get different points" works here. Delete `"scramble": true` and that advice becomes a no-op, which is why the first line of output above matters more than it looks.

## Scientific checkpoint

Write a short decision note before opening the answers:

1. When is grid the better choice?
2. When is sobol the better choice?
3. Which would you pick for a 6-10D study, and what number would you cite to defend it?
4. Your collaborator hands you a 5-input grid spec with 4 levels per input and asks whether
   it is enough to fit a surrogate. Which of the two quantities from Step 4 do you compute,
   and which do you refuse to answer from?

<details>
<summary><b>Compare your answer</b></summary>

1. **Grid** when `d ≤ 3` (Step 4: at `d = 2` the fill distances differ by a few percent, so
   the extra machinery buys nothing); when a downstream tool needs a regular lattice —
   contour plots, tensor-product interpolation, ANOVA on factor levels; and when you want to
   read a specific factor's effect by holding the others fixed, which a space-filling design
   cannot give you.
2. **Sobol** when `d ≥ 4`; when N is fixed by your compute budget and you want the best use
   of it; when the points will train a surrogate, which wants interior coverage rather than
   corners; and when the job may be pre-empted, because a truncated Sobol design is still
   space-filling while a truncated grid is a slice of one corner.
3. **Sobol**, and the number to cite is the face fraction `1 - 2⁻ᵈ`: at `d = 6` a 4-level grid
   is 4096 runs with 98.4% of them on the boundary. The discrepancy ratio from Step 4 is the
   second number.
4. **Fill distance** — it bounds how far the surrogate ever has to interpolate, which is the
   question actually being asked. **Refuse to answer from discrepancy**: Step 4 showed 8 Sobol
   points beating a 4096-point grid on discrepancy at `d = 6`, so a good discrepancy is no
   evidence that a design is informative enough to fit anything. (The honest full answer also
   needs the response surface's roughness, which no DOE metric knows.)

</details>

## Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `UserWarning: The balance properties of Sobol' points require n to be a power of 2` | `n_points` is not a power of two | Ask for 8, 16, 32… You still get *exactly* the count you asked for — a request for 9 returns 9 — but the dyadic stratification that makes Sobol *Sobol* is only partly delivered, so you paid for uniformity you did not fully receive. |
| Grid explodes to thousands of points | Cartesian product — cost is exponential in dimension | 3 levels × 6 inputs is 729 runs; 4 levels is 4096. Run `bayesmm plan` first — it is free and instant. |
| You changed `seed` and got the same Sobol points back | `scramble` is false (the planner's default) and scipy ignores `seed` for an unscrambled engine | Set `"scramble": true` in `design.sobol`. Step 5 demonstrates both cases. |
| A grid sweep sampled outside the box the spec declares | `plan_grid_points` never reads `io_schema.inputs[].support`; `bayesmm validate` does not check it either | Nothing will tell you — check your levels yourself, or use `sobol`, whose box *is* the support. Step 1 reproduces the failure. |
| Sobol points look clumped in a 2-D plot | You're projecting >2 dimensions onto 2 | Space-filling is a property of the full space; any 2-D shadow of it can look clustered. |

**Carry this into T5:** DOE choice determines what your surrogate can learn — and the
surrogate will not warn you when the answer is "not enough". T5 fits the backend named
`pymc_gp`, which despite its name is **not** a Gaussian process: `surrogates/backends.py`
builds `beta ~ Normal(0, 2)`, `intercept ~ Normal(0, 2)`, `mu = intercept + x @ beta` —
Bayesian *linear* regression, with no kernel anywhere in the package. A real GP would widen
its error bars over a region your design never sampled, and that widening is your warning
light. A linear model has no such reflex: it extends the fitted plane outward with
undiminished confidence, arbitrarily far. So a design with a hole in it does not yield a
visibly uncertain surrogate — it yields a confident one that happens to be wrong, which is
the more dangerous of the two failures. That is why where you put your points is not a
formality.

## Final check: T4's DOE claims, not T4's files

This cell re-derives both designs from the specs and asserts the things the notebook actually
taught: that 9 points were planned for each; that the Sobol design is neither the lattice nor
stuck on the boundary while the grid puts 8 of 9 points there; that at equal budget Sobol's
discrepancy is at least 2x better in 2-D and 10x better at `d = 4`; and that both sweeps
executed *exactly* the planned points, so the plan-is-a-contract claim in Step 2 is tested
rather than asserted. A notebook that ran but taught nothing fails here.

In [ ]:
# Self-check: assert the DOE science, not the existence of files.
import csv as _csv
import itertools as _it
import json as _json
import warnings as _w

import numpy as _np
from scipy.stats.qmc import Sobol as _Sobol
from scipy.stats.qmc import discrepancy as _disc

from bayesian_metamodeling.designs.planner import plan_points as _plan
from bayesian_metamodeling.spec import ModelSpec as _MS

_gspec = _json.loads((root / "tutorials/specs/model.toy.grid.json").read_text())
_sspec = _json.loads((root / "tutorials/specs/model.toy.sobol.json").read_text())
with _w.catch_warnings():
    _w.simplefilter("ignore")
    _sob = _plan(_MS.model_validate(_sspec))
_gri = _plan(_MS.model_validate(_gspec))
_S = _np.array([[p["a"], p["b"]] for p in _sob])
_G = _np.array([[p["a"], p["b"]] for p in _gri])

# (1) Both designs planned at equal budget. Also proves the plotting path is alive:
#     Step 3 draws these very arrays, so a dead design source cannot pass here.
assert _S.shape == (9, 2) and _G.shape == (9, 2), (
    f"planned {_G.shape} grid / {_S.shape} sobol points; expected 9 x 2 each."
)

# (2) Sobol is not the lattice, and does not live on the boundary the grid lives on.
_lat = _np.array([[a, b] for a in (0.0, 1.0, 2.0) for b in (0.0, 1.0, 2.0)])
_hits = int((_np.abs(_S[:, None, :] - _lat[None, :, :]).max(axis=2) < 1e-9).any(axis=1).sum())
assert _hits == 0, f"{_hits} sobol point(s) landed on the 3x3 lattice — the designs coincide."
_edge_s = int(((_S <= 0.0) | (_S >= 2.0)).any(axis=1).sum())
_edge_g = int(((_G <= 0.0) | (_G >= 2.0)).any(axis=1).sum())
assert _edge_g == 8, f"grid should put 8 of 9 points on the boundary; got {_edge_g}."
assert _edge_s <= 1, f"{_edge_s} sobol points on the boundary — Step 3's contrast is gone."

# (3) The tutorial's thesis, at equal budget. Observed ratio ~6x here and >=3.8x across 40
#     scramble seeds, so 2x leaves real margin for a scipy or seed change.
_dg, _ds = float(_disc(_G / 2.0)), float(_disc(_S / 2.0))
assert _ds < _dg / 2.0, f"sobol discrepancy {_ds:.5f} not < half of grid's {_dg:.5f}."

# (4) The curse-of-dimensionality claim, measured rather than quoted. Observed ratio ~700x.
_lv = _np.linspace(0.0, 1.0, 4)
_G4 = _np.array(list(_it.product(*[_lv] * 4)))
with _w.catch_warnings():
    _w.simplefilter("ignore")
    _S4 = _Sobol(d=4, scramble=True, seed=123).random(len(_G4))
assert float(_disc(_S4)) < float(_disc(_G4)) / 10.0, (
    "at d=4 and equal N, sobol's discrepancy is not 10x better than the grid's — "
    "the Step 4 table no longer supports the Step 4 conclusion."
)

# (5) The plan is a contract: the sweeps executed exactly the points that were planned.
def _newest_rows(store):
    _sweeps = sorted((root / store).glob("*/sweep_rows.csv"), key=lambda p: p.stat().st_mtime)
    assert _sweeps, f"no sweep under {store} — Step 2's `bayesmm run` didn't produce data."
    with open(_sweeps[-1]) as fh:
        return list(_csv.DictReader(fh)), _sweeps[-1]


_report = []
for _store, _planned, _label in (
    ("tmp/tutorials/toy_store/sweeps", _gri, "grid"),
    ("tmp/tutorials/toy_store_sobol/sweeps", _sob, "sobol"),
):
    _rows, _path = _newest_rows(_store)
    _ok = [r for r in _rows if r.get("status") == "success"]
    assert len(_ok) == 9, f"{_label}: {len(_ok)} successful rows in {_path.name}, expected 9."
    _ran = _np.array(sorted((float(r["a"]), float(r["b"])) for r in _ok))
    _want = _np.array(sorted((p["a"], p["b"]) for p in _planned))
    _dev = float(_np.abs(_ran - _want).max())
    assert _dev < 1e-9, (
        f"{_label}: the sweep did not execute the planned design "
        f"(max coordinate deviation {_dev:.3e}). `plan` and `run` disagree."
    )
    _report.append(f"{_label} 9/9 rows == plan (max dev {_dev:.1e}) at {_path.relative_to(root)}")

print(f"\n[T4 self-check OK] grid boundary points {_edge_g}/9, sobol {_edge_s}/9; "
      f"discrepancy grid {_dg:.5f} vs sobol {_ds:.5f} ({_dg / _ds:.1f}x)")
for _line in _report:
    print(f"[T4 self-check OK] {_line}")